In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Dataset (Assuming pima-indians-diabetes.csv is in the same directory)

In [2]:
%pwd

'/opt/app-root/src/ai267-sa/ch05'

In [3]:
# 1. Load Dataset (Assuming pima-indians-diabetes.csv is in the same directory)
# Columns: Pregnancies, Glucose, BP, Skin, Insulin, BMI, Pedigree, Age, Outcome
# url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

df = pd.read_csv("diabetes-model/pima-indians-diabetes.data.csv", names=names)

In [4]:
# Replace raw Age with a binary protected attribute for TrustyAI SPD
df["AgeOver50"] = (df["Age"] > 50).astype(np.float32)
df = df.drop(columns=["Age"])

feature_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "AgeOver50",
]

In [5]:
X_df = df[feature_cols].copy()
y = df["Outcome"].astype(np.int64).values

In [6]:
# Scale only the continuous columns; keep AgeOver50 as literal 0/1
scale_cols = [c for c in feature_cols if c != "AgeOver50"]
scaler = StandardScaler()
X_df.loc[:, scale_cols] = scaler.fit_transform(X_df[scale_cols])

X = X_df.values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

/tmp/ipykernel_6930/3382338024.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.63994726 -0.84488505  1.23388019 -0.84488505 -1.14185152  0.3429808
 -0.25095213  1.82781311 -0.54791859  1.23388019  0.04601433  1.82781311
  1.82781311 -0.84488505  0.3429808   0.93691372 -1.14185152  0.93691372
 -0.84488505 -0.84488505 -0.25095213  1.23388019  0.93691372  1.53084665
  2.12477957  1.82781311  0.93691372 -0.84488505  2.7187125   0.3429808
  0.3429808  -0.25095213 -0.25095213  0.63994726  1.82781311  0.04601433
  2.12477957  1.53084665 -0.54791859  0.04601433 -0.25095213  0.93691372
  0.93691372  1.53084665  0.93691372 -1.14185152 -0.84488505 -0.54791859
  0.93691372  0.93691372 -0.84488505 -0.84488505  0.3429808   1.23388019
  0.93691372 -0.84488505  0.93691372 -1.14185152 -1.14185152 -1.14185152
 -0.54791859  1.23388019  0.3429808  -0.54791859  0.93691372  0.3429808
 -1.14185152 -0.54791859 -0.84488505  0.04601

In [7]:
# 2. Define the Model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2) # 2 Outputs for CrossEntropy (Class 0 and Class 1)
        )

    def forward(self, x):
        logits = self.net(x)
        # We output BOTH labels and probabilities to satisfy TrustyAI and Performance metrics
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(probs, dim=1) 
        return labels, probs

model = DiabetesModel()

In [8]:
# 3. Quick Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

X_t = torch.FloatTensor(X_train)
y_t = torch.LongTensor(y_train)

for epoch in range(100):
    optimizer.zero_grad()
    outputs_labels, outputs_probs = model(X_t)
    # Loss uses raw logits or specialized layers; here we simplify
    loss = criterion(model.net(X_t), y_t) 
    loss.backward()
    optimizer.step()

In [9]:
# 4. Export to ONNX
model.eval()
dummy_input = torch.randn(1, len(feature_cols))

torch.onnx.export(
    model,
    dummy_input,
    "diabetes.onnx",
    verbose=False,
    input_names=["dense_input"],
    output_names=["label", "probabilities"],
    dynamic_axes={
        "dense_input": {0: "batch_size"},
        "label": {0: "batch_size"},
        "probabilities": {0: "batch_size"},
    },
    opset_version=11,
)

print("✅ Model saved as diabetes.onnx")
print("Features used:", feature_cols)

✅ Model saved as diabetes.onnx
Features used: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'AgeOver50']
